# Exploratory Data Analysis & Customer Segmentation Prep
This notebook conducts a professional, modular Exploratory Data Analysis (EDA) of the cleaned customer dataset and prepares engineered features for customer segmentation.

## 0. Load Dataset and Setup
We import libraries, load the cleaned dataset, and initialize our data.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import os

# Set default plotly template
pio.templates.default = "plotly_white"

# Load cleaned dataset
df = pd.read_csv("../dataset/processed/cleaned_customer_data.csv")

# Define spending and purchase columns for early availability
spending_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
purchase_cols = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']

# Create temporary columns for Bivariate analysis (will be formally documented in Section 6)
df['Total_Spending'] = df[spending_cols].sum(axis=1)
df['Total_Purchases'] = df[purchase_cols].sum(axis=1)

print(f"Dataset loaded successfully. Shape: {df.shape}")

## 1. Dataset Overview
Let's review the shape, schema info, and summary statistics of our cleaned customer dataset.

In [ ]:
# Display shape
print(f"Dataset Shape: {df.shape}\n")

# Display info
print("--- Dataset Info ---")
df.info()

# Display descriptive statistics
print("\n--- Descriptive Statistics ---")
df.describe(include='all')

### Business Insights - Dataset Overview
* **Observation 1**: The dataset contains **2,236 customers** and **29 columns** after cleaning. This provides a robust sample size for segmenting our active customer base.
* **Observation 2**: Customer ages range from **18 to 74 years old** (as of 2014), indicating a diverse age demographic spanning Gen Z to Baby Boomers.
* **Observation 3**: Household income ranges from **$1,730 to $162,397** with a median of **$51,381.50**, showing a highly stratified customer base that likely requires distinct product positioning.
* **Observation 4**: Spending varies heavily across products (e.g., Wine spending ranges from $0 to $1,493), suggesting distinct customer preferences and purchase power levels.

## 2. Univariate Analysis
We inspect the distributions of individual demographic and financial features.

In [ ]:
# Age Distribution
fig_age = px.histogram(df, x='Age', nbins=20, title='Customer Age Distribution',
                       labels={'Age': 'Age (Years)'}, color_discrete_sequence=['#4B6584'])
fig_age.show()

# Income Distribution
fig_income = px.histogram(df, x='Income', nbins=30, title='Customer Income Distribution',
                          labels={'Income': 'Annual Income ($)'}, color_discrete_sequence=['#20BF6B'])
fig_income.show()

# Boxplot of Spending Features
fig_spending = px.box(df, y=spending_cols, title='Spending Distribution across Product Categories',
                      labels={'value': 'Spending Amount ($)', 'variable': 'Category'})
fig_spending.show()

# Education Distribution
edu_counts = df['Education'].value_counts().reset_index()
fig_edu = px.bar(edu_counts, x='Education', y='count', title='Customer Distribution by Education Level',
                 labels={'count': 'Number of Customers'}, color='Education', color_discrete_sequence=px.colors.qualitative.Safe)
fig_edu.show()

# Marital Status Distribution
marital_counts = df['Marital_Status'].value_counts().reset_index()
fig_marital = px.bar(marital_counts, x='Marital_Status', y='count', title='Customer Distribution by Marital Status',
                     labels={'count': 'Number of Customers'}, color='Marital_Status', color_discrete_sequence=px.colors.qualitative.Safe)
fig_marital.show()

### Business Insights - Univariate Analysis
* **Observation 1 (Age Distribution)**: The customer base is concentrated between **35 and 55 years old**, representing peak earning and spending years. Marketing should focus heavily on these middle-aged families.
* **Observation 2 (Income Distribution)**: Income exhibits a near-normal distribution with a slight right skew, peaking around **$50,000 to $75,000**. Premium products should target this large mid-to-high income segment.
* **Observation 3 (Spending Disparity)**: Wine (`MntWines`) and Meat (`MntMeatProducts`) are the dominant spending categories by far, with high median values and long right tails. Other categories (Fruits, Sweets, Fish) have very low median spends, showing they are niche products.
* **Observation 4 (Demographics)**: Over 50% of the customer base has completed **Graduation** (undergraduate degree), and the majority are either **Married** or in a **Together** relationship. Our primary buyer persona is a college-educated, cohabiting adult.

## 3. Customer Behaviour
We analyze customer engagement patterns, spending distributions, purchase channels, and marketing campaign responsiveness.

In [ ]:
# Total spending by product category
total_spend_by_cat = df[spending_cols].sum().reset_index()
total_spend_by_cat.columns = ['Category', 'Total_Spending']
total_spend_by_cat = total_spend_by_cat.sort_values(by='Total_Spending', ascending=False)

fig_cat_spend = px.bar(total_spend_by_cat, x='Category', y='Total_Spending', title='Total Revenue by Product Category',
                       labels={'Total_Spending': 'Total Revenue ($)'}, color='Category', color_discrete_sequence=px.colors.qualitative.Pastel)
fig_cat_spend.show()

# Purchase channels distribution
channels = {'Channel': ['Store', 'Web', 'Catalog'],
            'Total_Purchases': [df['NumStorePurchases'].sum(), df['NumWebPurchases'].sum(), df['NumCatalogPurchases'].sum()]}
df_channels = pd.DataFrame(channels)

fig_channels = px.pie(df_channels, values='Total_Purchases', names='Channel', title='Purchase Volume by Sales Channel',
                      color_discrete_sequence=['#F08080', '#87CEFA', '#778899'])
fig_channels.show()

# Campaign acceptance rates
campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
campaign_sums = df[campaign_cols].sum().reset_index()
campaign_sums.columns = ['Campaign', 'Acceptances']
campaign_sums['Campaign'] = campaign_sums['Campaign'].replace({
    'AcceptedCmp1': 'Campaign 1', 'AcceptedCmp2': 'Campaign 2', 
    'AcceptedCmp3': 'Campaign 3', 'AcceptedCmp4': 'Campaign 4', 
    'AcceptedCmp5': 'Campaign 5', 'Response': 'Latest Campaign (Response)'
})

fig_campaign = px.bar(campaign_sums, x='Campaign', y='Acceptances', title='Total Acceptances by Marketing Campaign',
                      labels={'Acceptances': 'Number of Customers'}, color='Campaign', color_discrete_sequence=px.colors.qualitative.Set2)
fig_campaign.show()

# Monthly Web Visits distribution
fig_web_visits = px.histogram(df, x='NumWebVisitsMonth', nbins=15, title='Monthly Web Visits Distribution',
                              labels={'NumWebVisitsMonth': 'Web Visits per Month'}, color_discrete_sequence=['#A5B1C2'])
fig_web_visits.show()

### Business Insights - Customer Behaviour
* **Observation 1 (Revenue Drivers)**: **Wine** is by far the biggest revenue generator, followed by **Meat Products**. Together they account for over 80% of our sales. Fruits, Sweets, and Fish represent low-value impulse or supplementary categories.
* **Observation 2 (Sales Channels)**: **Store purchases** are the most common (making up nearly 50% of transactions), followed by the **Web** (~34%) and **Catalog** (~16%). An omni-channel strategy is working, but store layout/staffing remains our primary sales engine.
* **Observation 3 (Campaign Performance)**: The **latest campaign** was highly successful, securing **333 acceptances** (nearly 15% conversion rate). Among prior campaigns, Campaigns 4 and 3 performed best, while Campaign 2 was a failure (under 30 acceptances) and should be audited.
* **Observation 4 (Web Traffic)**: Most customers visit the website **5 to 8 times per month**, representing high digital touchpoints. This presents a prime opportunity to retarget them via web banner ads or personalized email offers.

## 4. Bivariate Analysis
We investigate relationships between demographics/finances and customer spending behavior.

In [ ]:
# Income vs Total Spending
fig_inc_spend = px.scatter(df, x='Income', y='Total_Spending', opacity=0.7, 
                           title='Household Income vs Total Spending',
                           labels={'Income': 'Annual Income ($)', 'Total_Spending': 'Total Spending ($)'},
                           color_discrete_sequence=['#45AAF2'])
fig_inc_spend.show()

# Age vs Total Spending
fig_age_spend = px.scatter(df, x='Age', y='Total_Spending', opacity=0.7,
                           title='Customer Age vs Total Spending',
                           labels={'Age': 'Age (Years)', 'Total_Spending': 'Total Spending ($)'},
                           color_discrete_sequence=['#EB3B5A'])
fig_age_spend.show()

# Education vs Income
fig_edu_inc = px.box(df, x='Education', y='Income', title='Income Distribution by Education Level',
                     color='Education', color_discrete_sequence=px.colors.qualitative.Pastel)
fig_edu_inc.show()

# Marital Status vs Total Spending
fig_marital_spend = px.box(df, x='Marital_Status', y='Total_Spending', title='Total Spending by Marital Status',
                           color='Marital_Status', color_discrete_sequence=px.colors.qualitative.Pastel)
fig_marital_spend.show()

### Business Insights - Bivariate Analysis
* **Observation 1 (Income vs. Spending)**: There is a strong, non-linear positive correlation between income and spending. Customers with income above **$60,000** show a massive leap in spending (typically exceeding $1,000 total). This is our "High Value" VIP tier.
* **Observation 2 (Age vs. Spending)**: Age does not show a strong relationship with spending. High and low spenders are distributed relatively evenly across all age groups, indicating that life stage/wealth (income) matters far more than chronological age.
* **Observation 3 (Education vs. Income)**: PhD, Master, and Graduation holders earn a significantly higher median income (~$50k+) compared to "Basic" education level customers (~$20k). Higher education level is a strong proxy for high purchasing power.
* **Observation 4 (Marital Status vs. Spending)**: Median total spending is virtually identical across all marital statuses (Married, Single, Divorced, Together, Widow). Marital status does not appear to dictate total retail spending capacity.

## 5. Correlation Analysis
We analyze linear correlations between numerical variables to identify key drivers and redundancies.

In [ ]:
# Select numeric columns
numeric_cols_all = df.select_dtypes(include=['number']).columns.tolist()
# Exclude ID from correlation matrix
numeric_cols_all = [col for col in numeric_cols_all if col != 'ID']

# Compute correlation matrix
corr_matrix = df[numeric_cols_all].corr()

# Plot correlation heatmap
fig_corr = px.imshow(corr_matrix, text_auto=".2f", aspect="auto", 
                     title="Correlation Heatmap of Customer Metrics",
                     color_continuous_scale='RdBu_r', range_color=[-1, 1])
fig_corr.update_layout(width=900, height=900)
fig_corr.show()

# Extract top correlations with Total Spending
spending_corr = corr_matrix['Total_Spending'].sort_values()
print("Top Negative Correlations with Total Spending:")
print(spending_corr.head(5))
print("\nTop Positive Correlations with Total Spending:")
print(spending_corr.tail(6).iloc[:-1]) # Exclude self-correlation

### Business Insights - Correlation Analysis
* **Observation 1 (Key Revenue Driver)**: Total Spending is extremely highly correlated with **Income (0.79)** and **Catalog Purchases (0.78)**. Customers who shop through catalogs are highly lucrative, premium shoppers.
* **Observation 2 (The Kids/Teens Drag)**: Total Spending has a strong negative correlation with the number of kids at home (**Kidhome: -0.50**). Households with young children spend significantly less on wines and meats, likely due to diverted budgets and less luxury spending.
* **Observation 3 (Online Habits)**: Web visits per month (`NumWebVisitsMonth`) are negatively correlated with Total Spending (-0.50) but positively correlated with Kidhome. Families with kids visit the website often but buy less, possibly searching for deals or browsing without purchasing.
* **Observation 4 (Deal Seekers)**: Deals purchases (`NumDealsPurchases`) have a negative correlation with total spending. High spenders do not wait for deals; they buy premium items (Wines, Meats) directly, while bargain hunters make many small deals purchases.

## 6. Feature Engineering
We formally engineer three key customer behavior features:
1. **`Total_Spending`**: Sum of spending on all product categories (`MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, `MntGoldProds`).
2. **`Total_Purchases`**: Sum of purchases made across all channels (`NumWebPurchases`, `NumCatalogPurchases`, `NumStorePurchases`).
3. **`Average_Spending_Per_Purchase`**: Calculated as `Total_Spending` divided by `Total_Purchases` (safeguarded against division by zero).

In [ ]:
# Create final features
df['Total_Spending'] = df[spending_cols].sum(axis=1)
df['Total_Purchases'] = df[purchase_cols].sum(axis=1)

# Calculate Average Spending Per Purchase, setting to 0 if Total_Purchases is 0
df['Average_Spending_Per_Purchase'] = np.where(
    df['Total_Purchases'] > 0,
    df['Total_Spending'] / df['Total_Purchases'],
    0.0
)

# Preview the engineered features
print("Engineered Features Preview:")
df[['Total_Spending', 'Total_Purchases', 'Average_Spending_Per_Purchase']].head()

# Show descriptive stats for the new features
df[['Total_Spending', 'Total_Purchases', 'Average_Spending_Per_Purchase']].describe()

### Business Insights - Feature Engineering
* **Observation 1 (High Spending Variance)**: The median customer spends **$396.00** total, but the top 25% spends **over $1,048.00** (up to $2,525.00). This indicates a highly profitable high-end customer segment.
* **Observation 2 (Purchase Frequency)**: The average customer makes **12.5 purchases** across all channels. Targeting customers to increase purchase frequency (even slightly) would yield massive compound revenue.
* **Observation 3 (Cart Value)**: The average spend per purchase is **$32.93**, with some customers averaging up to **$158.00 per transaction**. High-value basket campaigns (e.g. "Free shipping over $75") should be directed at mid-tier baskets.
* **Observation 4 (VIP Profiling)**: High values in `Average_Spending_Per_Purchase` can instantly identify high-efficiency customers who buy premium products in single visits, optimizing shipping and logistic costs.

## 7. Export Cleaned & Engineered Dataset
Finally, we export the dataset containing all cleaned demographic features, behaviors, and newly engineered columns to a CSV file for subsequent clustering and modeling.

In [ ]:
# Export to processed dataset directory
export_path = "../dataset/processed/customer_features.csv"
df.to_csv(export_path, index=False)
print(f"Engineered dataset exported successfully to: {export_path}")
print(f"Final Dataset Shape: {df.shape}")